## Build the maximum-clique model


## Connect the fresh cost model to the quantum layers

The editable root below uses the regenerated seven-vertex diagonal Hamiltonian
and all ten QAOA layers. FlagQuantum supplies Pauli evolution and repeated
mixers; the classical model chooses the conflict penalty and the optimizer
chooses twenty angles. The identity cost term is retained rather than silently
removed from the coherent phase convention.

Try a provided nonzero schedule before launching another optimizer run. The
400-query ceiling and 10000 draws per evaluation are preserved. A low sampled
energy is not proof of a maximum clique; check feasibility of the decoded nodes.
The 218-point upstream trace and external CX/depth figures remain historical,
not newly measured external performance or an identical optimizer trajectory.


In [ ]:
from IPython.display import Image, display

from examples.synthesis.tutorials.technology_demonstrations.qaoa.qaoa_demonstration.overview.main import (
    historical_energies,
    report,
    save_figure,
)
from examples.synthesis.tutorials.technology_demonstrations.qaoa.qaoa_demonstration.overview.model import (
    LAYERS,
    SHOTS,
    WIDTH,
    build_model,
)
from examples.synthesis.tutorials.technology_demonstrations.qaoa.qaoa_demonstration.overview.workflow import (
    MAX_ITERATIONS,
    MAX_QUERIES,
    run_workflow,
)

model = build_model()
print("Nonedges of the seven-vertex graph:", model.conflicts)


## Retain the original graph and sample budget


In [ ]:
assert (WIDTH, LAYERS, SHOTS) == (7, 10, 10000)
print("Graph vertices and edges:", WIDTH, len(model.edges))

from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
    suzuki_trotter,
)
from examples.synthesis.tutorials.technology_demonstrations.qaoa.qaoa_demonstration.overview.model import (
    initial_parameters,
)

hamiltonian = model.hamiltonian


@qfunc
def clique_program(params: list[float], target: Output[QArray[QBit]]) -> None:
    allocate(7, target)
    hadamard_transform(target)
    for layer in range(10):
        suzuki_trotter(hamiltonian, params[2 * layer], 1, 1, target)
        for bit in range(7):
            RX(params[2 * layer + 1], target[bit])


preview = synth.synthesize(
    clique_program,
    classical_inputs={"params": initial_parameters(model)},
    max_qubits=7,
    max_workspace=0,
    max_operations=8192,
    memory_limit_bytes=1024**2,
)
print("Editable ten-layer resources:", preview.report["resources"])

import torch
import flagquantum as fq
from examples.synthesis.compare_states import compare_states

preview_angles = initial_parameters(model)
preview_reference = fq.Circuit(7, dtype=torch.complex128)
for bit in range(7):
    preview_reference.h(bit)
for layer in range(10):
    preview_reference.any(
        *range(7),
        matrix=torch.diag(
            torch.exp(
                -1j
                * preview_angles[2 * layer]
                * torch.tensor(model.energies, dtype=torch.float64)
            )
        )
    )
    for bit in range(7):
        preview_reference.rx(bit, theta=preview_angles[2 * layer + 1])
compare_states(preview, preview_reference, ("target",))


## Profile the public CX/U3 conversion


In [ ]:
print("Compare native execution and separate CX/U3 resource estimates.")


## Run the native sampled optimizer


In [ ]:
print("Iteration limit and objective query ceiling:", MAX_ITERATIONS, MAX_QUERIES)
run = run_workflow()


## Inspect the complete diagonal Hamiltonian


In [ ]:
print("Walsh Pauli terms, including the identity:", len(model.coefficients))
print("Energy bounds:", min(model.energies), max(model.energies))


## Read the original Qiskit reference


In [ ]:
reference = historical_energies()
assert len(reference) == 218


## Read the native trace length


In [ ]:
print("Measured FlagQuantum objective queries:", len(run.trace))


## Compare published and measured resource metrics


In [ ]:
print("Source Qiskit depth and CX count:", 1646, 1660)
print("Measured native CX/U3 estimate:", run.resources)


## Identify the fixed reference data


In [ ]:
print("The upstream reference trace is retained as historical data.")


## Inspect the finite-shot optimizer outcomes


In [ ]:
report(run)


## Read the lowest observed mean energy


In [ ]:
print("Lowest observed mean energy:", run.mean)
print("Selected sampled clique:", run.nodes, "Feasible:", run.clique)


## Plot both traces


In [ ]:
display(Image(filename=str(save_figure(run))))
